In [2]:
import rasterio as rio
import numpy as np

import os
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
data_path = Path(os.environ["DATA_PATH"])
out_path = data_path / "generated"

In [4]:
ZONE = "PER+Coronel Portillo"

In [78]:
arr = []
with rio.open(out_path / "small" / "raster" / "built_binary" / f"{ZONE}.tif") as ds:
    profile = ds.profile
    for year in range(2000, 2021, 5):
        idx = year - 1999
        arr.append(ds.read(idx))

arr = np.array(arr)

In [81]:
((arr[0] == 1) & (arr[1] == 0)).sum()

np.int64(4438)

In [40]:
arrs_progressive = [arr[0]]
for i in range(1, arr.shape[0]):
    diff = arr[i].astype(int) - arr[i - 1].astype(int)
    diff[diff < 0] = 0
    diff = diff.astype(np.uint8)
    arrs_progressive.append(diff)

arrs_progressive = np.array(arrs_progressive)

In [41]:
profile_mod = profile.copy()
profile_mod["count"] = arrs_progressive.shape[0]
with rio.open("./test.tif", "w", **profile_mod) as ds:
    for i, band in enumerate(arrs_progressive):
        ds.write(band, i + 1)

In [48]:
test = (arrs_progressive * np.array(list(range(1, arrs_progressive.shape[0] + 1))).reshape(-1, 1, 1))

In [75]:
i = 0
np.bitwise_and(arrs_progressive[0] == 1,  arrs_progressive[2] == 1).sum()

np.int64(1471)

In [ ]:
test = (arrs_progressive * np.array(list(range(1, arrs_progressive.shape[0] + 1))).reshape(-1, 1, 1)).sum(axis=0)

profile_mod = profile.copy()
profile_mod["count"] = 1
with rio.open("./test2.tif", "w", **profile_mod) as ds:
    ds.write(test, 1)